# Multi-agent avec CrewAI

CrewAI permet de faire collaborer plusieurs agents IA, chacun avec un **role**, un **objectif**, et eventuellement un **modele different**.

Dans cet exemple :
- **Agent Chercheur** (Groq, tres rapide) : rassemble des faits bruts sur un sujet.
- **Agent Redacteur** (Claude, plus rigoureux) : reprend ces faits et redige une reponse finale claire et structuree.

Les deux agents travaillent en **sequence** : le resultat du premier sert d'entree au second (pipeline d'agents specialises).

In [ ]:
!pip install -q crewai

## Configuration des cles API

Reutilise les memes cles que dans `test_gemini_groq.ipynb` (Secrets Colab `ANTHROPIC_API_KEY` et `GROQ_API_KEY`, sinon saisie manuelle).

In [ ]:
import os
from getpass import getpass

def get_key(env_name, prompt):
    key = None
    try:
        from google.colab import userdata
        key = userdata.get(env_name)
    except Exception:
        pass
    if not key:
        key = getpass(prompt)
    os.environ[env_name] = key
    return key

anthropic_key = get_key('ANTHROPIC_API_KEY', 'Entre ta cle API Anthropic: ')
groq_key = get_key('GROQ_API_KEY', 'Entre ta cle API Groq: ')

## Definir les modeles (un par agent)

CrewAI utilise la convention LiteLLM : `"<fournisseur>/<nom_du_modele>"`.

In [ ]:
from crewai import LLM

groq_llm = LLM(
    model='groq/llama-3.1-8b-instant',
    api_key=groq_key,
)

claude_llm = LLM(
    model='anthropic/claude-haiku-4-5',
    api_key=anthropic_key,
)

## Definir les agents et les taches

In [ ]:
from crewai import Agent, Task, Crew, Process

chercheur = Agent(
    role='Chercheur',
    goal="Rassembler rapidement des faits precis et pertinents sur le sujet demande",
    backstory="Tu es un chercheur efficace qui produit des listes de faits bruts, sans fioritures.",
    llm=groq_llm,
    verbose=True,
)

redacteur = Agent(
    role='Redacteur',
    goal="Transformer des faits bruts en une reponse claire, structuree et bien ecrite",
    backstory="Tu es un redacteur rigoureux qui organise l'information de facon pedagogique.",
    llm=claude_llm,
    verbose=True,
)

sujet = "les avantages et inconvenients de l'energie solaire pour une maison individuelle en France"

tache_recherche = Task(
    description=f"Liste 5 a 8 faits factuels et verifiables sur : {sujet}. Format : liste a puces, une phrase par fait.",
    expected_output="Une liste a puces de faits bruts.",
    agent=chercheur,
)

tache_redaction = Task(
    description=(
        "A partir des faits fournis par le chercheur, redige une reponse structuree en 3 parties : "
        "avantages, inconvenients, et une recommandation finale en une phrase."
    ),
    expected_output="Une reponse structuree en 3 parties (avantages / inconvenients / recommandation).",
    agent=redacteur,
    context=[tache_recherche],
)

## Lancer la crew (execution sequentielle)

In [ ]:
crew = Crew(
    agents=[chercheur, redacteur],
    tasks=[tache_recherche, tache_redaction],
    process=Process.sequential,
    verbose=True,
)

resultat = crew.kickoff()

print('\n=== Reponse finale ===')
print(resultat)